# Notebook 2 — MGEN Script Generation

Loads a saved traffic-generation run and converts the time-aligned burst
tables into MGEN `ON`/`OFF` event scripts.

```
traffic_profiles/run_<apps>_<ts>/bursts/  
        ↓
  ★ THIS code ★
        ↓
traffic_profiles/run_<apps>_<ts>/mgen_scripts/
        ↓
Deployment
```


## Cell 1 — Load Previous Run

In [1]:
import math
import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── helpers ───────────────────────────────────────────────────────────────────

def find_project_root() -> Path:
    current = Path.cwd()
    for cand in [current, *current.parents]:
        if (cand / "data").is_dir() and (cand / "artifacts").is_dir():
            return cand
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT         = find_project_root()
TRAFFIC_PROFILES_DIR = PROJECT_ROOT / "traffic_profiles"

# ── list available runs ───────────────────────────────────────────────────────

print("="*72)
print("  LOAD TRAFFIC GENERATION RUN")
print("="*72)
print(f"\n  Available runs in: traffic_profiles/\n")

available_runs = sorted(
    d for d in TRAFFIC_PROFILES_DIR.iterdir()
    if d.is_dir() and d.name.startswith("run_")
)

if not available_runs:
    raise FileNotFoundError(
        f"No runs found in {TRAFFIC_PROFILES_DIR}. "
        "Run Notebook 1 first."
    )

for idx, run_dir in enumerate(available_runs):
    cfg = run_dir / "config.json"
    if cfg.exists():
        with open(cfg) as f:
            c = json.load(f)
        apps     = ", ".join(c.get("apps", ["?"]))
        duration = c.get("simulation_duration", "?")
        n_ue     = c.get("n_ue", "?")
    else:
        apps = duration = n_ue = "?"
    print(f"  [{idx}] {run_dir.name}")
    print(f"       apps={apps}  duration={duration}s  n_ue={n_ue}")
    print()

# ── select run ────────────────────────────────────────────────────────────────

# OPTION 1 — auto: latest run
RUN_DIR = available_runs[-1]


# OPTION 2 — by index (uncomment)
# RUN_INDEX = 0
# RUN_DIR = available_runs[RUN_INDEX]


# OPTION 3 — by name (uncomment)
# RUN_DIR = TRAFFIC_PROFILES_DIR / "run_..."


print(f"  Selected: {RUN_DIR.name}")
MGEN_OUTPUT_DIR = RUN_DIR / "mgen_scripts"
MGEN_OUTPUT_DIR.mkdir(exist_ok=True)

# ── load config ───────────────────────────────────────────────────────────────

with open(RUN_DIR / "config.json") as f:
    config = json.load(f)

# ── validation gate ───────────────────────────────────────────────────────────
# Notebook 1 saves runs even when validation fails (so the data is preserved).
# Set ALLOW_INVALID_RUN = True only if intentionally want to generate
# MGEN scripts from a run that did not fully pass validation checks.


ALLOW_INVALID_RUN = False

_validation = config.get("validation", {})
_passed     = _validation.get("passed", True)   # True = assume ok if key absent
if not _passed and not ALLOW_INVALID_RUN:
    _errors  = _validation.get("errors", [])
    _preview = "\n".join(f"  • {e}" for e in _errors[:5])
    _extra   = f"  ... and {len(_errors)-5} more\n" if len(_errors) > 5 else ""
    _msg     = (
        f"Run '{RUN_DIR.name}' failed Notebook 1 validation"
        f" ({len(_errors)} error(s)):\n"
        f"{_preview}\n{_extra}"
        f"\nFix the run in Notebook 1, "
        f"or set ALLOW_INVALID_RUN=True to force script generation."
    )
    raise ValueError(_msg)
elif _passed:
    print("  ✅  Notebook 1 validation passed")
else:
    print("  ⚠️   Notebook 1 validation FAILED — proceeding anyway (ALLOW_INVALID_RUN=True)")

N_UE                = config["n_ue"]
SIMULATION_DURATION = config["simulation_duration"]
APPS                = config["apps"]
DN_IP               = config["network"]["dn_ip"]
UE_IP_PREFIX        = config["network"]["ue_ip_prefix"]
UE_IP_START         = config["network"]["ue_ip_start"]
DL_PORT             = config["network"]["dl_port"]
UL_PORT             = config["network"]["ul_port"]

print(f"\n  Apps     : {', '.join(APPS)}")
print(f"  Duration : {SIMULATION_DURATION}s")
print(f"  UEs      : {N_UE}")
print(f"  DN IP    : {DN_IP}")
print(f"  UE IPs   : {UE_IP_PREFIX}{UE_IP_START} – "
      f"{UE_IP_PREFIX}{UE_IP_START + N_UE - 1}")
print(f"  DL port  : {DL_PORT}  (DN → UE)")
print(f"  UL port  : {UL_PORT}  (UE → DN)")

# ── load UE assignments ───────────────────────────────────────────────────────

ue_assignments_df = pd.read_csv(RUN_DIR / "ue_flow_assignments.csv")
ue_flow_assignments: Dict[str, Dict] = {
    row.ue_name: {"class": row.ue_class}
    for row in ue_assignments_df.itertuples()
}

# ── load burst parquets ───────────────────────────────────────────────────────

bursts_dir    = RUN_DIR / "bursts"
ue_burst_data: Dict[str, Dict] = {}

print(f"\n  Loading burst data:")

for row in ue_assignments_df.itertuples():
    ue_name = row.ue_name
    ue_burst_data[ue_name] = {}

    for app in APPS:
        dl_file = bursts_dir / f"{ue_name}_{app}_dl_bursts.parquet"
        ul_file = bursts_dir / f"{ue_name}_{app}_ul_bursts.parquet"

        dl_df = pd.read_parquet(dl_file) if dl_file.exists() else pd.DataFrame()
        ul_df = pd.read_parquet(ul_file) if ul_file.exists() else pd.DataFrame()

        ue_burst_data[ue_name][app] = {"dl": dl_df, "ul": ul_df}

        dl_tag = "(projected)" if "packets_export" in dl_df.columns else "(raw)"
        ul_tag = "(projected)" if "packets_export" in ul_df.columns else "(raw)"
        print(f"    {ue_name}/{app}: {len(dl_df)} DL {dl_tag} + "
              f"{len(ul_df)} UL {ul_tag}")

print(f"\n  ✅  Cell 1 complete — "
      f"working with {RUN_DIR.relative_to(PROJECT_ROOT)}")

  LOAD TRAFFIC GENERATION RUN

  Available runs in: traffic_profiles/

  [0] run_filimo_igap_youtube_telegram_aparat_20260410_154510
       apps=filimo, igap, youtube, telegram, aparat  duration=600s  n_ue=6

  [1] run_filimo_igap_youtube_telegram_aparat_20260411_131932
       apps=filimo, igap, youtube, telegram, aparat  duration=600s  n_ue=6

  [2] run_filimo_igap_youtube_telegram_aparat_20260411_140454
       apps=filimo, igap, youtube, telegram, aparat  duration=600.0s  n_ue=6

  Selected: run_filimo_igap_youtube_telegram_aparat_20260411_140454
  ✅  Notebook 1 validation passed

  Apps     : filimo, igap, youtube, telegram, aparat
  Duration : 600.0s
  UEs      : 6
  DN IP    : 192.168.70.163
  UE IPs   : 12.1.1.1 – 12.1.1.6
  DL port  : 5000  (DN → UE)
  UL port  : 6000  (UE → DN)

  Loading burst data:
    ue1/filimo: 168 DL (projected) + 9 UL (projected)
    ue1/igap: 10 DL (projected) + 7 UL (projected)
    ue1/youtube: 218 DL (projected) + 617 UL (projected)
    ue1/telegram: 

## Cell 2 — Generate MGEN Scripts

Converts burst tables to MGEN `ON`/`OFF` event lines and writes one
script per UE (DL receiver + UL sender) plus the two DN scripts.


In [2]:
import re
MAX_PKT = 1472   # UDP payload ceiling
MIN_PKT = 64     # UDP payload floor

# ── Flow ID namespace constants ─────────────────────────────────────────────
# Hierarchy: UE → app → original flow → burst counter
# Each level gets its own stride so IDs never collide across levels:
#   unique_flow_id = ue_namespace
#                  + app_idx  × APP_ID_STRIDE
#                  + flow_id  × FLOW_ID_STRIDE
#                  + burst_counter


UE_ID_STRIDE            = 100_000_000
APP_ID_STRIDE           =  10_000_000
FLOW_ID_STRIDE          =      10_000

# Source port bands — stable per (app, original_flow) for readability

SRC_PORT_BASE           = 10_000
SRC_PORT_STRIDE_PER_APP =  5_000

# Built at runtime from the loaded APPS list 

APP_ID_MAP: Dict[str, int] = {app: i for i, app in enumerate(APPS)}

# Guard: verify source ports stay within valid UDP range (1024–65535).

_max_src_port = (SRC_PORT_BASE
                 + (len(APPS) - 1) * SRC_PORT_STRIDE_PER_APP
                 + SRC_PORT_STRIDE_PER_APP - 1)
assert _max_src_port <= 65535, (
    f"Source port ceiling {_max_src_port} exceeds UDP maximum 65535. "
    f"Too many apps ({len(APPS)}) for current SRC_PORT_STRIDE_PER_APP. "
    f"Reduce SRC_PORT_STRIDE_PER_APP or increase SRC_PORT_BASE."
)
print(f"  Source port range: {SRC_PORT_BASE} – {_max_src_port}  ✅")


def burst_to_mgen_params(burst: pd.Series) -> tuple:
    """
    Return (packets_export, pkt_size_export, pps, on_dur) for one burst.

    Prefers projected columns from the audit notebook.
    Falls back to the byte-preserving formula if they are absent.
    """
    on_dur = max(0.001, float(burst["on_dur_s"]))

    if "packets_export" in burst.index and "pkt_size_export" in burst.index:
        
        # ── projected path (preferred) ────────────────────────────────────
        
        pkts     = max(1, int(burst["packets_export"]))
        pkt_size = int(burst["pkt_size_export"])
        pkt_size = max(MIN_PKT, min(pkt_size, MAX_PKT))
        pps      = float(burst["pps"]) if "pps" in burst.index else pkts / on_dur
    else:
        
        # ── fallback: byte-preserving formula  ─────────────
        
        raw_bytes = max(1, float(burst["bytes"]))
        raw_pkts  = max(1, int(burst["packets"]))
        pkts      = max(raw_pkts, math.ceil(raw_bytes / MAX_PKT))
        pkt_size  = math.ceil(raw_bytes / pkts)
        pkt_size  = max(MIN_PKT, min(pkt_size, MAX_PKT))
        pps       = pkts / on_dur

    return pkts, pkt_size, round(pps, 3), on_dur


def bursts_to_mgen_events(
    burst_df: pd.DataFrame,
    dst_ip: str,
    dst_port: int,
    ue_namespace: int,
    app_name: str,
) -> List[str]:
    
    """
    Convert a burst DataFrame to a list of MGEN event strings.

    Each burst becomes exactly one ON / OFF pair with a fully namespaced
    unique flow ID, preventing collisions across UEs, apps, flows, and
    bursts even when the same synthetic_flow_id appears in multiple apps.

    Flow ID hierarchy:
        unique_flow_id = ue_namespace
                       + app_idx  × APP_ID_STRIDE
                       + flow_id  × FLOW_ID_STRIDE
                       + burst_counter

    Source port hierarchy:
        src_port = SRC_PORT_BASE
                 + app_idx × SRC_PORT_STRIDE_PER_APP
                 + (flow_id % SRC_PORT_STRIDE_PER_APP)
        Stable per (app, original_flow) — readable in packet captures.
    """
    
    if burst_df.empty:
        return []

    app_idx       = APP_ID_MAP[app_name]
    app_namespace = app_idx * APP_ID_STRIDE

    events: List[str] = []

    for burst_counter, (_, burst) in enumerate(burst_df.iterrows()):
        base_flow_id   = int(burst["synthetic_flow_id"])

        unique_flow_id = (
            ue_namespace
            + app_namespace
            + base_flow_id * FLOW_ID_STRIDE
            + burst_counter
        )

        pkts, pkt_size, pps, on_dur = burst_to_mgen_params(burst)

        t_on  = float(burst["absolute_start_time"])
        t_off = float(burst["absolute_end_time"])

        src_port = (
            SRC_PORT_BASE
            + app_idx * SRC_PORT_STRIDE_PER_APP
            + (base_flow_id % SRC_PORT_STRIDE_PER_APP)
        )

        events.append(
            f"{t_on:.6f} ON {unique_flow_id} UDP "
            f"SRC {src_port} DST {dst_ip}/{dst_port} "
            f"PERIODIC [{pps:.3f} {pkt_size}]"
        )
        events.append(f"{t_off:.6f} OFF {unique_flow_id}")

    events.sort(key=lambda x: float(x.split()[0]))
    return events



# ── generate ──────────────────────────────────────────────────────────────────

print("="*72)
print("  GENERATING MGEN SCRIPTS")
print("="*72)
print(f"\n  Output: {MGEN_OUTPUT_DIR.relative_to(PROJECT_ROOT)}\n")

mgen_scripts: Dict[str, Dict] = {}
manifest_rows: List[Dict]     = []

# byte fidelity tracking (global)

total_orig_bytes = 0
total_proj_bytes = 0


_ue_names_sorted = sorted(
    ue_burst_data.keys(),
    key=lambda s: [int(t) if t.isdigit() else t
                   for t in re.split(r"(\d+)", s)]
)
for ue_idx, ue_name in enumerate(_ue_names_sorted):
    bursts_by_app = ue_burst_data[ue_name]
    ue_ip         = f"{UE_IP_PREFIX}{UE_IP_START + ue_idx}"

    # UE namespace: each UE occupies a non-overlapping block of IDs
    
    ue_namespace = ue_idx * UE_ID_STRIDE

    print(f"  {ue_name.upper()}  "
          f"(IP: {ue_ip}  class: "
          f"{ue_flow_assignments[ue_name]['class']}  "
          f"namespace: {ue_namespace:,}):")

    all_dl_events: List[str] = []
    all_ul_events: List[str] = []

    for app in APPS:
        dl_df = bursts_by_app[app]["dl"]
        ul_df = bursts_by_app[app]["ul"]

        dl_ev = bursts_to_mgen_events(dl_df, ue_ip,  DL_PORT, ue_namespace, app)
        ul_ev = bursts_to_mgen_events(ul_df, DN_IP,  UL_PORT, ue_namespace, app)

        all_dl_events.extend(dl_ev)
        all_ul_events.extend(ul_ev)

        # byte fidelity for projected rows
        
        for df in [dl_df, ul_df]:
            if df.empty:
                continue
            total_orig_bytes += df["bytes"].sum()
            if "packets_export" in df.columns and "pkt_size_export" in df.columns:
                total_proj_bytes += (
                    df["packets_export"] * df["pkt_size_export"]
                ).sum()
            else:
                # recompute with fallback formula for tracking
                pkts = df["bytes"].apply(
                    lambda b: max(1, math.ceil(b / MAX_PKT))
                )
                pksz = (df["bytes"] / pkts).apply(math.ceil)
                total_proj_bytes += (pkts * pksz).sum()

        print(f"    {app:<12} DL {len(dl_ev):>5} events   "
              f"UL {len(ul_ev):>5} events")

    # sort merged events by time
    
    all_dl_events.sort(key=lambda x: float(x.split()[0]))
    all_ul_events.sort(key=lambda x: float(x.split()[0]))

    # ── write UE DL receiver script ───────────────────────────────────────
    
    dl_rx_path = MGEN_OUTPUT_DIR / f"{ue_name}_dl_rx.mgn"
    with open(dl_rx_path, "w") as fh:
        fh.write(f"# MGEN Downlink Receiver — {ue_name.upper()}\n")
        fh.write(f"# Receives from DN ({DN_IP}) on port {DL_PORT}\n")
        fh.write(f"# UE IP : {ue_ip}\n")
        fh.write(f"# Class : {ue_flow_assignments[ue_name]['class']}\n")
        fh.write(f"# Apps  : {', '.join(APPS)}\n\n")
        fh.write(f"0.0 LISTEN UDP {DL_PORT}\n")

    # ── write UE UL sender script ─────────────────────────────────────────
    
    ul_tx_path = MGEN_OUTPUT_DIR / f"{ue_name}_ul_tx.mgn"
    with open(ul_tx_path, "w") as fh:
        fh.write(f"# MGEN Uplink Sender — {ue_name.upper()}\n")
        fh.write(f"# Sends to DN ({DN_IP}:{UL_PORT})\n")
        fh.write(f"# UE IP      : {ue_ip}\n")
        fh.write(f"# Class      : {ue_flow_assignments[ue_name]['class']}\n")
        fh.write(f"# Apps       : {', '.join(APPS)}\n")
        fh.write(f"# UL events  : {len(all_ul_events)}\n")
        fh.write(f"# UE namespace: {ue_namespace:,}\n\n")
        for ev in all_ul_events:
            fh.write(ev + "\n")

    print(f"    ✅  {dl_rx_path.name}  "
          f"({len(all_dl_events):,} DL events)")
    print(f"    ✅  {ul_tx_path.name}  "
          f"({len(all_ul_events):,} UL events)\n")

    manifest_rows.append({
        "ue_name"      : ue_name,
        "ue_ip"        : ue_ip,
        "ue_class"     : ue_flow_assignments[ue_name]["class"],
        "dl_rx_script" : dl_rx_path.name,
        "ul_tx_script" : ul_tx_path.name,
        "n_dl_events"  : len(all_dl_events),
        "n_ul_events"  : len(all_ul_events),
        "ue_namespace" : ue_namespace,
    })

    mgen_scripts[ue_name] = {
        "dl_events"  : all_dl_events,
        "ul_events"  : all_ul_events,
        "ue_ip"      : ue_ip,
        "ue_namespace": ue_namespace,
    }

# ── DN downlink sender (all UEs combined) ─────────────────────────────────────

all_dn_dl: List[str] = []
for scripts in mgen_scripts.values():
    all_dn_dl.extend(scripts["dl_events"])
all_dn_dl.sort(key=lambda x: float(x.split()[0]))

dn_dl_path = MGEN_OUTPUT_DIR / "dn_dl_tx.mgn"
with open(dn_dl_path, "w") as fh:
    fh.write("# MGEN Downlink Sender — DN → ALL UEs\n")
    fh.write(f"# Source      : {DN_IP}\n")
    ue_ips = ", ".join(
        f"{UE_IP_PREFIX}{UE_IP_START+i}" for i in range(N_UE)
    )
    fh.write(f"# Destinations: {ue_ips}\n")
    fh.write(f"# Apps        : {', '.join(APPS)}\n")
    fh.write(f"# Duration    : {SIMULATION_DURATION}s\n")
    fh.write(f"# DL events   : {len(all_dn_dl):,}\n\n")
    for ev in all_dn_dl:
        fh.write(ev + "\n")

# ── DN uplink receiver ────────────────────────────────────────────────────────

dn_ul_path = MGEN_OUTPUT_DIR / "dn_ul_rx.mgn"
with open(dn_ul_path, "w") as fh:
    fh.write("# MGEN Uplink Receiver — DN ← ALL UEs\n")
    fh.write(f"# Listens on port {UL_PORT}\n")
    fh.write(f"# Sources: {ue_ips}\n\n")
    fh.write(f"0.0 LISTEN UDP {UL_PORT}\n")

# ── manifest ──────────────────────────────────────────────────────────────────

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MGEN_OUTPUT_DIR / "manifest.csv", index=False)

# ── byte fidelity ─────────────────────────────────────────────────────────────

byte_error_pct = (
    100 * abs(total_proj_bytes - total_orig_bytes) / total_orig_bytes
    if total_orig_bytes > 0 else 0
)

# ── summary ───────────────────────────────────────────────────────────────────

total_dl = sum(r["n_dl_events"] for r in manifest_rows)
total_ul = sum(r["n_ul_events"] for r in manifest_rows)

print("="*72)
print("  MANIFEST")
print("="*72)
print(manifest_df.to_string(index=False))

print(f"\n{'='*72}")
print(f"  SUMMARY")
print(f"{'='*72}")
print(f"""
  Generated files : {len(manifest_rows)*2 + 2}
    {len(manifest_rows)} × UE DL receivers   (ue*_dl_rx.mgn)
    {len(manifest_rows)} × UE UL senders     (ue*_ul_tx.mgn)
    1   DN DL sender      (dn_dl_tx.mgn)
    1   DN UL receiver    (dn_ul_rx.mgn)

  Total DL events : {total_dl:,}
  Total UL events : {total_ul:,}
  Combined        : {total_dl+total_ul:,}

  Byte fidelity   : {byte_error_pct:.3f}% error  """
      f"(≤ 1 B per packet from ceil rounding)"
)
print(f"  Saved to : {MGEN_OUTPUT_DIR.relative_to(PROJECT_ROOT)}")
print()
print("  ✅  Cell 2 complete — ready (deployment)")

  GENERATING MGEN SCRIPTS

  Output: traffic_profiles/run_filimo_igap_youtube_telegram_aparat_20260411_140454/mgen_scripts

  UE1  (IP: 12.1.1.1  class: heavy  namespace: 0):
    filimo       DL   336 events   UL    18 events
    igap         DL    20 events   UL    14 events
    youtube      DL   436 events   UL  1234 events
    telegram     DL   876 events   UL   296 events
    aparat       DL   158 events   UL    22 events
    ✅  ue1_dl_rx.mgn  (1,826 DL events)
    ✅  ue1_ul_tx.mgn  (1,584 UL events)

  UE2  (IP: 12.1.1.2  class: heavy  namespace: 100,000,000):
    filimo       DL   144 events   UL    24 events
    igap         DL    14 events   UL     8 events
    youtube      DL   184 events   UL   826 events
    telegram     DL  1084 events   UL   354 events
    aparat       DL   154 events   UL     8 events
    ✅  ue2_dl_rx.mgn  (1,580 DL events)
    ✅  ue2_ul_tx.mgn  (1,220 UL events)

  UE3  (IP: 12.1.1.3  class: light  namespace: 200,000,000):
    filimo       DL     4 event